# 02 · Campaign — scaffold near the NA + LigandMPNN design around it

**Standard slot:** *design campaign.* **For Project 23 this is the core:** build the NA-binder pool (D2):
- **RFdiffusion** — scaffold protein backbones **docked against the nucleic-acid target** (hold the NA
  as context so the backbone forms a complementary recognition surface).
- **LigandMPNN (NA-aware)** — design sequences for each backbone **with the NA in context** (the central
  step). We also design a **ProteinMPNN (NA-blind)** set on the *same* backbones for the nb04 benchmark.

Then model every design as a **protein–NA complex** (`pae_interaction` is the key complex metric).

> **Compute honesty:** a real campaign wants an **A100** (Colab Pro+ or a cluster) for RFdiffusion near
> the NA and for protein–NA complex modeling. **LigandMPNN/ProteinMPNN are CPU-cheap** — the modeling
> is the bottleneck. Free **T4** = a *small fallback* (few backbones, small modeling batch). The cells
> below run on the deterministic **mock** backend so the plumbing executes anywhere; the real calls +
> A100 notes are shown alongside. Run `00_setup.ipynb` first.
>
> **Protein–NA design is newer and harder than protein–protein.** RFdiffusion's nucleic-acid support is
> evolving — **verify the current protocol/commit** (version-verify cell below).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change!)

The NA-binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log it).
LigandMPNN's nucleic-acid support and RFdiffusion's NA protocol both evolve, so this check matters here.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   LigandMPNN   https://github.com/dauparas/LigandMPNN        # NA-aware sequence design (CENTRAL); pin <commit>
#   RFdiffusion  https://github.com/RosettaCommons/RFdiffusion # scaffold near the NA; pin <commit>
#   Boltz        https://github.com/jwohlwend/boltz            # protein-NA complex modeling (Boltz-2); pin <commit>
#   ColabFold    https://github.com/sokrypton/ColabFold        # AF2 complex fallback; pin <commit>
PINNED = {
    "LigandMPNN":  "https://github.com/dauparas/LigandMPNN",
    "RFdiffusion": "https://github.com/RosettaCommons/RFdiffusion",
    "Boltz":       "https://github.com/jwohlwend/boltz",
    "ColabFold":   "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:12s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:12s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("Verify LigandMPNN's NA (ligand_mpnn) model + RFdiffusion's nucleic-acid protocol specifically — they evolve.")

## 1 · Define the campaign

Same target motif as notebook 01. Set honest campaign sizes; the cells run on `mock` so they execute
anywhere. On Colab (A100) switch the `TOOL_*` to the real backends — and **shrink the numbers on a
T4** (few backbones, a small modeling batch). We generate **both** sequence designers (LigandMPNN
NA-aware + ProteinMPNN NA-blind) on the **same backbones** so the nb04 head-to-head isolates the value
of nucleic-acid conditioning.

In [ ]:
import na_binder_tools as nbt
import pandas as pd

NA_TYPE = "DNA"
MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE)   # EXAMPLE — replace with your verified target motif
SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)

# Honest campaign sizes: scaffold many backbones near the NA, design several sequences each.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BACKBONES   = 40      # -> hundreds of RFdiffusion backbones near the NA on A100; few on T4
N_SEQ_PER_BB  = 2       # -> 4-8 LigandMPNN sequences per backbone on Colab

TOOL_SCAFFOLD = "mock"  # -> "rfdiffusion" on Colab (A100)
TOOL_SEQ      = "mock"  # -> "ligandmpnn" / "proteinmpnn" on Colab
TOOL_MODEL    = "mock"  # -> "boltz" / "af3" on Colab (A100)

print(f"backbones (RFdiffusion): n={N_BACKBONES} tool={TOOL_SCAFFOLD}")
print(f"sequences/backbone     : n={N_SEQ_PER_BB} tool={TOOL_SEQ} (LigandMPNN NA-aware + ProteinMPNN NA-blind)")
print(f"complex model          : tool={TOOL_MODEL}")
print("motif :", MOTIF, " scrambled:", SCRAMBLED)

## 2 · Scaffold backbones near the nucleic acid (RFdiffusion)

Diffuse protein backbones docked against the NA target, holding the nucleic acid as fixed context so
the backbone forms a complementary recognition surface (a helix into the major groove, a sheet, ...).
Sequence is **not** designed yet. On A100 this is hundreds of backbones; the `mock` backend returns
deterministic `SYNTHETIC` backbones with placeholder sequences.

In [ ]:
# Real call (Colab, A100): nbt.scaffold_near_na(MOTIF, n=N_BACKBONES, tool="rfdiffusion", na_type=NA_TYPE)
#   keep the NA chain fixed as context; verify RFdiffusion's current nucleic-acid protocol/commit.
backbones = nbt.scaffold_near_na(MOTIF, n=N_BACKBONES, tool=TOOL_SCAFFOLD, na_type=NA_TYPE)
print(f"scaffolded {len(backbones)} backbones near the {NA_TYPE} target (tool={TOOL_SCAFFOLD}; SYNTHETIC if mock)")
print("example:", backbones[0].design_id, " length=", backbones[0].length,
      " (sequence undesigned ->", backbones[0].seq_tool + ")")

## 3 · Design sequences around the NA — LigandMPNN (NA-aware) **and** ProteinMPNN (NA-blind)

The central step. **LigandMPNN** conditions on the nucleic-acid atoms, so its interface residues are
chosen to read the bases/backbone. **ProteinMPNN** sees only the protein backbone (NA-blind) — we design
it on the *same* backbones purely as the benchmark baseline. (Ligand)MPNN is CPU-cheap; this is not the
bottleneck. Then model each design as a protein–NA complex (the slow step on Colab).

In [ ]:
# Real call (Colab): nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool="ligandmpnn", na_type=NA_TYPE,
#   seq_tool="ligandmpnn", temperature=0.1)  with the NA in context (model_type=ligand_mpnn).
#   For the NA-BLIND baseline: tool="proteinmpnn", seq_tool="proteinmpnn".
ligand_designs, protein_designs = [], []
for bb in backbones:
    ligand_designs  += nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool=TOOL_SEQ,
                                         na_type=NA_TYPE, seq_tool="ligandmpnn")
    protein_designs += nbt.ligandmpnn_na(bb, MOTIF, n=N_SEQ_PER_BB, tool=TOOL_SEQ,
                                         na_type=NA_TYPE, seq_tool="proteinmpnn")

# Model each (protein, NA) complex -> pae_interaction, plddt, scrmsd.
nbt.score_designs(ligand_designs,  na=MOTIF, tool=TOOL_MODEL)
nbt.score_designs(protein_designs, na=MOTIF, tool=TOOL_MODEL)

# Specificity: intended motif vs scrambled motif (the protein-NA-specific layer).
nbt.add_specificity(ligand_designs,  MOTIF, scrambled=SCRAMBLED, tool=TOOL_MODEL)
nbt.add_specificity(protein_designs, MOTIF, scrambled=SCRAMBLED, tool=TOOL_MODEL)

print(f"LigandMPNN (NA-aware) designs: {len(ligand_designs)}")
print(f"ProteinMPNN (NA-blind) designs: {len(protein_designs)}")
print("example LigandMPNN:", ligand_designs[0].design_id,
      "pae_interaction=", ligand_designs[0].pae_interaction,
      "specificity_score=", ligand_designs[0].specificity_score)

## 4 · Assemble + persist both pools

Write one tidy CSV per sequence designer (plus a combined one). These feed notebook 03 (the shared
filter) and notebook 04 (the LigandMPNN-vs-ProteinMPNN benchmark + specificity analysis). We add an
EXAMPLE `solubility` column so the physics layer has something to act on in the dry run — on Colab
these come from the real modeling/energetics; for `mock` they are SYNTHETIC.

In [ ]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, seq_tool=d.seq_tool,
            na_type=d.na_type, target_motif=d.target_motif, length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            dG_motif=d.dG_motif, dG_scrambled=d.dG_scrambled,
            specificity_score=d.specificity_score, is_specific=d.is_specific,
            solubility=0.3,        # EXAMPLE_DATA placeholder so Layer 3 (physics) runs in the dry run
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_lig = pool_to_df(ligand_designs);  df_lig.to_csv("results/ligandmpnn_designs.csv", index=False)
df_pro = pool_to_df(protein_designs); df_pro.to_csv("results/proteinmpnn_designs.csv", index=False)
combined = pd.concat([df_lig, df_pro], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/ligandmpnn_designs.csv  ", df_lig.shape, " (NA-aware — your real candidates)")
print("wrote results/proteinmpnn_designs.csv ", df_pro.shape, " (NA-blind baseline — for the benchmark)")
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

## D2 checklist
- [ ] Backbones scaffolded **near the nucleic acid** (hundreds on A100; few on T4), NA held as context.
- [ ] **LigandMPNN (NA-aware)** sequences designed around the NA (the central step) — your real candidate pool.
- [ ] **ProteinMPNN (NA-blind)** designed on the *same* backbones as the benchmark baseline.
- [ ] Every design modeled as a protein–NA complex (`pae_interaction` parsed) **and** scored for motif-vs-scrambled specificity; both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured (esp. LigandMPNN NA model + RFdiffusion NA protocol); 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the pools.